In [38]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from functions import *
import sys, os

## Sundstrom et al. 2025
https://arxiv.org/pdf/2505.04577 

https://github.com/msundstrom33/ComparingActiveLearningMethods

In [39]:
### Taking the average of COPUS COPUS_codes_sundstrom_weir across multiple observations of the same class

path = 'Published Data/Sundstrom_2025_data/'


#load in COPUS COPUS_codes_weir
COPUS_codes_sundstrom = pd.read_csv(path + 'data/classroom-observations/LPA_Input.csv')
#delete rows where course is nan
COPUS_codes_sundstrom = COPUS_codes_sundstrom.dropna(subset=['Course'])
#create a new column that combines the course and method columns
COPUS_codes_sundstrom['Course'] = [str(COPUS_codes_sundstrom['Method'].iloc[i]) + '_Course' + str(int(COPUS_codes_sundstrom['Course'].iloc[i])) for i in range(len(COPUS_codes_sundstrom))]
#for each group of rows that have the same method and course, take the mean of the numeric values
numeric_cols = COPUS_codes_sundstrom.select_dtypes(include='number').columns
COPUS_codes_sundstrom = COPUS_codes_sundstrom.groupby(['Course'])[numeric_cols].mean().reset_index()

In [40]:
## Loading in concept inventory data

CI_statistics_sundstrom = pd.DataFrame()
CI_statistics_sundstrom['Course'] = COPUS_codes_sundstrom['Course']

max_scores = np.array([30, 30, 30, 30, 47, 0,##ISLE
 47, 47, 26, 26, 14, 30, 50, 47, 30, ##PI
 0, 0, 30, 30, 30, 30, 25, ##scale up
 26, 47, 33, 30, 26, 30, 30, 0]) ## Tutorials

courses = CI_statistics_sundstrom['Course'].values
for i in range(len(courses)):
    course = courses[i]

    #read in scores for the course
    file = path + 'data/rosters/Roster_' + course + '.csv'

    try:  #some COPUS courses do not have CI data
        CI_scores = pd.read_csv(path + 'data/rosters/Roster_' + course + '.csv')
        #input the class size
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'N'] = int(len(CI_scores))

        #We are only using matched scores, so we drop any rows with missing pre or post scores
        CI_scores.dropna(inplace = True)

        pretest_scores = CI_scores['PreScore'].values/max_scores[i]
        posttest_scores = CI_scores['PostScore'].values/max_scores[i]

        #add means and STDs to CI_statistics_sundstrom
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'PreMean'] = np.mean(pretest_scores)
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'PreStd'] = np.std(pretest_scores)
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'PostMean'] = np.mean(posttest_scores)
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'PostStd'] = np.std(posttest_scores)
    except:
        #If there is no CI data for the course, we remove it from the CI_statistics_sundstrom dataframe
        CI_statistics_sundstrom = CI_statistics_sundstrom[CI_statistics_sundstrom['Course'] != course]

#calculate effect sizes and add to CI_statistics_sundstrom
CI_statistics_sundstrom['Effect Size'] = [effect_size(CI_statistics_sundstrom['PreMean'].iloc[i],
                                         CI_statistics_sundstrom['PostMean'].iloc[i],
                                         CI_statistics_sundstrom['PreStd'].iloc[i],
                                         CI_statistics_sundstrom['PostStd'].iloc[i]) for i in range(len(CI_statistics_sundstrom))]

CI_statistics_sundstrom['Field'] = 'Physics'
astro_courses = ['Tutorials_Course2', 'Tutorials_Course6',]
CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'].isin(astro_courses), 'Field'] = 'Astronomy'

In [41]:
## combine COPUS COPUS_codes_weir with CI statistics
sundstrom_combined = pd.merge(COPUS_codes_sundstrom, CI_statistics_sundstrom, on='Course')
sundstrom_combined

,Course,CQ,MG,PQ,Lec,CG,WG,OG,SQ,N,PreMean,PreStd,PostMean,PostStd,Effect Size,Field
0,ISLE_Course1,0.178884,0.009804,0.811526,0.407428,0.162004,0.000000,0.145035,0.198617,150.0,0.278744,0.177654,0.441546,0.195802,0.870839,Physics
1,ISLE_Course2,0.056778,0.154216,0.222885,0.622976,0.000000,0.105691,0.034278,0.148590,16.0,0.413889,0.195533,0.461111,0.207201,0.234410,Physics
2,ISLE_Course3,0.253070,0.008772,0.704386,0.539474,0.128070,0.000000,0.100439,0.260526,118.0,0.345318,0.205252,0.423596,0.175190,0.410227,Physics
3,ISLE_Course4,0.000000,0.481863,0.910784,0.116667,0.000000,0.425000,0.102941,0.269608,15.0,0.423810,0.232457,0.645238,0.213663,0.991806,Physics
4,ISLE_Course5,0.000000,0.591475,0.859631,0.167351,0.000000,0.347034,0.244442,0.006410,32.0,0.248227,0.090269,0.476359,0.290706,1.059888,Physics
5,PeerInstruction_Course1,0.020202,0.058923,0.840132,0.886817,0.020202,0.000000,0.106643,0.230122,28.0,0.440729,0.184920,0.498480,0.203398,0.297105,Physics
6,PeerInstruction_Course2,0.339023,0.356330,0.661925,0.446420,0.266951,0.000000,0.071124,0.036036,263.0,0.252077,0.165272,0.359068,0.260662,0.490239,Physics
7,PeerInstruction_Course3,0.658333,0.430556,0.274444,0.489444,0.617222,0.027778,0.000000,0.000000,576.0,0.402808,0.194549,0.457979,0.255339,0.243057,Physics
8,PeerInstruction_Course4,0.471985,0.416904,0.178181,0.436135,0.404677,0.000000,0.025641,0.039530,201.0,0.527939,0.188314,0.574383,0.267998,0.200529,Physics
9,PeerInstruction_Course5,0.452727,0.283636,0.347879,0.471515,0.412727,0.000000,0.053333,0.055152,81.0,0.282143,0.203884,0.370238,0.217629,0.417773,Physics


## Weir et al. 2019

https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0220900

In [42]:
path = 'Published Data/Weir_2019_data/'

#read in COPUS codes
file = pd.ExcelFile(path + 'UBC_raw_COPUS.xlsx')
COPUS_codes_weir = pd.read_excel(file, sheet_name=file.sheet_names[-1])

#select only relevant columns, rename them, and convert values to percentages
COPUS_codes_weir = COPUS_codes_weir[['unique.section','num.students', 'ICQ', 'IPQ','SL', 'Ilec', 'SCG', 'SWG', 'SOG', 'SQ',  ]]
COPUS_codes_weir.columns = ['Course', 'N', 'CQ', 'PQ','L', 'Lec', 'CG', 'WG', 'OG', 'SQ', ]

COPUS_codes_weir.iloc[:, 2:] = COPUS_codes_weir.iloc[:, 2:].div(100)
COPUS_codes_weir['Field'] = 'Biology'


In [43]:
# load in CI statistics


CI_statistics_weir = pd.DataFrame()
#load raw scores, keep only relevant columns, convert to percentages
raw_scores = pd.read_excel(path + 'UBC_raw_scores.xlsx')
raw_scores.drop(columns=['course.level', 'term'], inplace=True)
raw_scores.iloc[:, 1:] = raw_scores.iloc[:, 1:].div(100)

course_list = raw_scores['unique.section'].unique()
for course in course_list:
    course_scores = raw_scores[raw_scores['unique.section'] == course]
    pretest_scores = course_scores['pre.score'].values
    posttest_scores = course_scores['post.score'].values
    CI_statistics_weir.loc[CI_statistics_weir.shape[0], 'Course'] = course
    CI_statistics_weir.loc[CI_statistics_weir.shape[0]-1, 'PreMean'] = np.mean(pretest_scores)
    CI_statistics_weir.loc[CI_statistics_weir.shape[0]-1, 'PreStd'] = np.std(pretest_scores)
    CI_statistics_weir.loc[CI_statistics_weir.shape[0]-1, 'PostMean'] = np.mean(posttest_scores)
    CI_statistics_weir.loc[CI_statistics_weir.shape[0]-1, 'PostStd'] = np.std(posttest_scores)

CI_statistics_weir['Effect Size'] = [effect_size(CI_statistics_weir['PreMean'].iloc[i],
                                         CI_statistics_weir['PostMean'].iloc[i],
                                         CI_statistics_weir['PreStd'].iloc[i],
                                         CI_statistics_weir['PostStd'].iloc[i])
                                     for i in range(len(CI_statistics_weir))]


In [44]:
## combine COPUS codes with CI statistics
weir_combined = pd.merge(COPUS_codes_weir, CI_statistics_weir, on = 'Course')
weir_combined.head()

,Course,N,CQ,PQ,L,Lec,CG,WG,OG,SQ,Field,PreMean,PreStd,PostMean,PostStd,Effect Size
0,1A,303,0.305250,0.344750,0.817000,0.532750,0.255500,0.263250,0.049750,0.211750,Biology,0.322528,0.165109,0.821839,0.178042,2.908094
1,1B,251,0.301000,0.196167,0.934500,0.875000,0.234500,0.062500,0.000000,0.094500,Biology,0.464645,0.246937,0.790243,0.177240,1.514889
2,1C,310,0.357667,0.303000,0.802667,0.565000,0.324667,0.187333,0.098667,0.156333,Biology,0.307833,0.165856,0.801228,0.203690,2.656393
3,1D,296,0.339000,0.408000,0.905667,0.677667,0.319000,0.120000,0.147667,0.285000,Biology,0.288056,0.176536,0.787386,0.185245,2.759605
4,2A,168,0.074000,0.320500,0.888000,0.333000,0.061500,0.000000,0.223500,0.159500,Biology,0.464674,0.117792,0.631114,0.124880,1.371146


## Connell et al 2017

https://www.lifescied.org/doi/full/10.1187/cbe.15-03-0062

In [45]:
#These values come from using webplot digitizer on from the graph  of COPUS codes in Connell et al. 2017

connell_combined = pd.DataFrame({
    'Course': ['Connell_ext', 'Connell_mod'],
    'L': [.65, .87],
    'Lec': [0.34, 0.75], #
    'PQ': [0.26, 0.28],#
    'MG': [0.31, 0.0], #
    'CQ': [0.0, 0.0], #not reported
    
    
    'CG': [0, 0], #not reported
    'WG': [0.4, 0.0],
    'OG': [0.12, 0.66], 
    'SQ': [0.12, 0.024], 
    
    'PreMean': [0.443, 0.429],
    'PostMean': [0.585, 0.546],
    'PreStd': [0.107, 0.104],
    'PostStd': [0.121, 0.122],
    'Class Size': [182, 172], #reported 316 took survey

    'N': [182, 172], #reported 316 took survey
    'Field': ['Biology', 'Biology'],
})

connell_combined['Effect Size'] = (connell_combined['PostMean'] - connell_combined['PreMean']) / np.sqrt(
    (connell_combined['PreStd']**2 + connell_combined['PostStd']**2) / 2)

connell_combined

,Course,L,Lec,PQ,MG,CQ,CG,WG,OG,SQ,PreMean,PostMean,PreStd,PostStd,Class Size,N,Field,Effect Size
0,Connell_ext,0.65,0.34,0.26,0.31,0.0,0,0.4,0.12,0.120,0.443,0.585,0.107,0.121,182,182,Biology,1.243272
1,Connell_mod,0.87,0.75,0.28,0.00,0.0,0,0.0,0.66,0.024,0.429,0.546,0.104,0.122,172,172,Biology,1.032130


## Combining Alll Datasets

In [48]:
all_data = pd.concat([sundstrom_combined, weir_combined, connell_combined], ignore_index=True)

#reorder columns
#we remove MG because it was only reported by Sundstrom not the others
all_data = all_data[['Course', 'N', 'Field', 'PreMean', 'PostMean', 'PreStd', 'PostStd', 'Effect Size',
                                        'L', 'Lec', 'CG', 'CQ', 'PQ', 'WG', 'OG', 'SQ', ]]

#When the mean of pretest scores is above 60, it imposes a ceiling effect on effect sizes
# and indicates that the CI may not be appropriate for that course, so we remove those courses
all_data = all_data[all_data['PreMean'] <= 0.6]

#saved combined data to csv
all_data.to_csv('Published Data/combined_COPUS_CI_data.csv', index=False)
all_data

,Course,N,Field,PreMean,PostMean,PreStd,PostStd,Effect Size,L,Lec,CG,CQ,PQ,WG,OG,SQ
0,ISLE_Course1,150.0,Physics,0.278744,0.441546,0.177654,0.195802,0.870839,NaN,0.407428,0.162004,0.178884,0.811526,0.000000,0.145035,0.198617
1,ISLE_Course2,16.0,Physics,0.413889,0.461111,0.195533,0.207201,0.234410,NaN,0.622976,0.000000,0.056778,0.222885,0.105691,0.034278,0.148590
2,ISLE_Course3,118.0,Physics,0.345318,0.423596,0.205252,0.175190,0.410227,NaN,0.539474,0.128070,0.253070,0.704386,0.000000,0.100439,0.260526
3,ISLE_Course4,15.0,Physics,0.423810,0.645238,0.232457,0.213663,0.991806,NaN,0.116667,0.000000,0.000000,0.910784,0.425000,0.102941,0.269608
4,ISLE_Course5,32.0,Physics,0.248227,0.476359,0.090269,0.290706,1.059888,NaN,0.167351,0.000000,0.000000,0.859631,0.347034,0.244442,0.006410
5,PeerInstruction_Course1,28.0,Physics,0.440729,0.498480,0.184920,0.203398,0.297105,NaN,0.886817,0.020202,0.020202,0.840132,0.000000,0.106643,0.230122
6,PeerInstruction_Course2,263.0,Physics,0.252077,0.359068,0.165272,0.260662,0.490239,NaN,0.446420,0.266951,0.339023,0.661925,0.000000,0.071124,0.036036
7,PeerInstruction_Course3,576.0,Physics,0.402808,0.457979,0.194549,0.255339,0.243057,NaN,0.489444,0.617222,0.658333,0.274444,0.027778,0.000000,0.000000
8,PeerInstruction_Course4,201.0,Physics,0.527939,0.574383,0.188314,0.267998,0.200529,NaN,0.436135,0.404677,0.471985,0.178181,0.000000,0.025641,0.039530
9,PeerInstruction_Course5,81.0,Physics,0.282143,0.370238,0.203884,0.217629,0.417773,NaN,0.471515,0.412727,0.452727,0.347879,0.000000,0.053333,0.055152
